In [1]:
!pip install langchain langchain-community
!pip install chromadb sentence-transformers
!pip install pypdf openai fastapi uvicorn
!pip install langchain-openai


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 195.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 237.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.6
    Uninstalling protobuf-4.25.6:
      Successfully uninstalled protobuf-4.25.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kfp-kubernetes 1.5.0 requires protobuf<5,>=4.21.1, but you have protobuf 6.33.5 which is incompatible.
kfp 2.12.1 requires protobuf<5,>=4.21.1, but you have protobuf 6.33.5 which is incompatible.
kfp-pipeline-spec 0.6.0 requires protobuf<5,>=4.21.1, but you have protobuf 6.33.5 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip in

In [3]:
import os

# Let's see what files are already there from workshop
for root, dirs, files in os.walk('/parasol-insurance'):
    for file in files:
        print(os.path.join(root, file))

In [4]:
import os

# Check what's in the parasol-insurance folder
for root, dirs, files in os.walk('parasol-insurance'):
    for file in files:
        print(os.path.join(root, file))

parasol-insurance/README.md
parasol-insurance/.gitignore
parasol-insurance/LICENSE
parasol-insurance/default-site.yml
parasol-insurance/app/README.md
parasol-insurance/app/package-lock.json
parasol-insurance/app/Pipfile
parasol-insurance/app/start-dev.sh
parasol-insurance/app/package.json
parasol-insurance/app/Containerfile
parasol-insurance/app/Pipfile.lock
parasol-insurance/app/claim-images/policy.json
parasol-insurance/app/claimdb/04-original_images_table_creation.sql
parasol-insurance/app/claimdb/01-claimdb_creation.sql
parasol-insurance/app/claimdb/02-claims_schema_creation.sql
parasol-insurance/app/claimdb/03-claims_table_creation.sql
parasol-insurance/app/claimdb/06-create_base_claims.sql
parasol-insurance/app/claimdb/05-processed_images_table_creation.sql
parasol-insurance/app/frontend/README.md
parasol-insurance/app/frontend/package-lock.json
parasol-insurance/app/frontend/.prettierrc
parasol-insurance/app/frontend/dr-surge.js
parasol-insurance/app/frontend/.gitignore
parasol-

In [5]:
# Let's read the existing claim data from the workshop
import json
import os

# Read the sample claims that are already there!
claims_path = 'parasol-insurance/lab-materials/03/claims'

for claim_file in os.listdir(claims_path):
    with open(f'{claims_path}/{claim_file}', 'r') as f:
        claim = json.load(f)
        print(f"=== {claim_file} ===")
        print(json.dumps(claim, indent=2))
        print()

=== claim1.json ===
{
  "claim-number": 1,
  "subject": "Claim for Recent Car Accident - Policy Number: AC-987654321",
  "content": "Dear Parasol Insurance,\n\nI hope this email finds you well. My name is Sarah Turner, and I am writing to file a claim for a recent car accident that occurred on January 2nd, 2024, at approximately 3:30 PM. My policy number is AC-987654321.\n\nThe accident took place at the intersection of Birch Street and Willow Avenue in the city of Evergreen. I was driving my vehicle, a black Toyota Camry with license plate number DEF-456, heading south on Birch Street. At the intersection, the traffic signal was green, and I proceeded through the intersection.\n\nAt the same time, another vehicle, a blue Chevrolet Traverse with license plate number GHI-789, was traveling west on Willow Avenue. Unfortunately, the driver failed to stop at the red traffic signal, resulting in a collision with the front passenger side of my vehicle.\n\nThe impact caused significant damage

In [6]:
import json
import os

# Load all claims
claims = []
claims_path = 'parasol-insurance/lab-materials/03/claims'

for claim_file in sorted(os.listdir(claims_path)):
    with open(f'{claims_path}/{claim_file}', 'r') as f:
        claim = json.load(f)
        claims.append(claim)
        print(f"✅ Loaded: {claim_file} - {claim['subject']}")

print(f"\nTotal claims loaded: {len(claims)}")

✅ Loaded: claim1.json - Claim for Recent Car Accident - Policy Number: AC-987654321
✅ Loaded: claim2.json - Urgent: Unacceptable Delay and Lack of Communication Regarding Claim #XYZ789
✅ Loaded: claim3.json - Urgent: Car Accident Claim Assistance Needed

Total claims loaded: 3


In [7]:
from langchain_community.llms import VLLMOpenAI

# This uses the LLM already running in OpenShift AI
# from the workshop!
llm = VLLMOpenAI(
    openai_api_key="EMPTY",
    openai_api_base="http://llm-service:8000/v1",
    model_name="granite",
    temperature=0,
    max_tokens=512
)

# Test it
response = llm.invoke("Say hello in one sentence")
print(response)

APIConnectionError: Connection error.

In [8]:
# Let's check what LLM URL was used in the workshop notebooks
import subprocess
result = subprocess.run(
    ['grep', '-r', 'inference_server_url\|openai_api_base\|llm-service\|model_name', 
     'parasol-insurance/lab-materials/03/'],
    capture_output=True, text=True
)
print(result.stdout[:3000])

parasol-insurance/lab-materials/03/03-02-summarization.ipynb:    "inference_server_url = \"http://granite-3-1-8b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080\"\n",
parasol-insurance/lab-materials/03/03-02-summarization.ipynb:    "    openai_api_base=f\"{inference_server_url}/v1\",\n",
parasol-insurance/lab-materials/03/03-02-summarization.ipynb:    "    model_name=\"granite-3-1-8b-instruct\",\n",
parasol-insurance/lab-materials/03/06/llm_usage.py:        openai_api_base=f"{INFERENCE_SERVER_URL}/v1",
parasol-insurance/lab-materials/03/06/llm_usage.py:        model_name="granite-3-1-8b-instruct",
parasol-insurance/lab-materials/03/03-04-comparing-model-servers.ipynb:    "inference_server_url = \"http://granite-3-1-8b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080\"\n",
parasol-insurance/lab-materials/03/03-04-comparing-model-servers.ipynb:    "    openai_api_base=f\"{inference_server_url}/v1\",\n",
parasol-insurance/lab-materials/03/03-04-comparing-model-servers.ipynb

In [9]:
from langchain_community.llms import VLLMOpenAI

# Use the exact URL from the workshop notebooks
inference_server_url = "http://granite-3-1-8b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080"

llm = VLLMOpenAI(
    openai_api_key="EMPTY",
    openai_api_base=f"{inference_server_url}/v1",
    model_name="granite-3-1-8b-instruct",
    temperature=0,
    max_tokens=512
)

# Test it
response = llm.invoke("Say hello in one sentence")
print(response)

 saying the name of language first.

¡Hola! Español. (Hello! Spanish.)

¡Hola! Español. (Hello! Spanish.)


In [10]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Step 1 - Convert claims to documents
documents = []
for claim in claims:
    doc = Document(
        page_content=claim['content'],
        metadata={
            "claim_number": claim['claim-number'],
            "subject": claim['subject']
        }
    )
    documents.append(doc)

# Step 2 - Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"✅ Created {len(chunks)} chunks from {len(documents)} claims")

# Step 3 - Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Step 4 - Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)
print("✅ Vector store created!")

✅ Created 21 chunks from 3 claims


/tmp/ipykernel_161/883899562.py:27: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RuntimeError: [91mYour system has an unsupported version of sqlite3. Chroma                     requires sqlite3 >= 3.35.0.[0m
[94mPlease visit                     https://docs.trychroma.com/troubleshooting#sqlite to learn how                     to upgrade.[0m

In [11]:
# Fix - use Milvus instead of Chroma
# Milvus is already running in OpenShift AI from the workshop!

from langchain_community.vectorstores import Milvus
from langchain_community.embeddings import HuggingFaceEmbeddings

# Embeddings already downloaded - reuse them
print("✅ Embeddings ready!")

# Create vector store using Milvus
# (already running in the cluster from workshop)
vectorstore = Milvus.from_documents(
    documents=chunks,
    embedding=embeddings,
    connection_args={
        "host": "milvus-service.ic-shared-milvus.svc.cluster.local",
        "port": "19530"
    },
    collection_name="insurance_claims"
)
print("✅ Milvus vector store created!")

✅ Embeddings ready!


Failed to create new connection using: 3128ba18e6474dd0a6ba87540ea44802


MilvusException: <MilvusException: (code=2, message=Fail connecting to server on milvus-service.ic-shared-milvus.svc.cluster.local:19530, illegal connection params or server unavailable)>

In [12]:
# Use FAISS - works completely locally, no server needed!
!pip install faiss-cpu -q

from langchain_community.vectorstores import FAISS

# Create vector store using FAISS locally
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✅ FAISS vector store created!")
print(f"✅ {vectorstore.index.ntotal} chunks indexed!")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ FAISS vector store created!
✅ 21 chunks indexed!


In [14]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Custom prompt for insurance claims
prompt_template = """
You are an expert insurance claims assistant for 
Parasol Insurance. Use the claim information below 
to answer the question accurately and helpfully.

Claim Information:
{context}

Question: {question}

Provide a clear, professional answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Build RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(
        search_kwargs={"k": 2}
    ),
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("✅ RAG chain is ready!")

✅ RAG chain is ready!


In [15]:
# Test with real questions about the actual claims!
questions = [
    "Who was at fault in Sarah Turner's accident?",
    "What is the policy number for claim 1?",
    "Which claimant seems most anxious or distressed?",
    "What damage was done in the accident on Birch Street?",
    "Which claim has the most urgent tone?"
]

print("🚗 InsureAI — Parasol Insurance Claims Assistant")
print("="*50)

for question in questions:
    print(f"\n❓ Question: {question}")
    result = rag_chain.invoke({"query": question})
    print(f"✅ Answer: {result['result']}")
    print(f"📄 Source: Claim #{result['source_documents'][0].metadata['claim_number']}")
    print("-"*50)

🚗 InsureAI — Parasol Insurance Claims Assistant

❓ Question: Who was at fault in Sarah Turner's accident?
✅ Answer: 
Based on the information provided, Samantha Reynolds was at fault in Sarah Turner's accident. She ran a red light at the intersection of Maple Street and Oak Avenue in Rivertown, causing the collision with Sarah Turner's vehicle.
📄 Source: Claim #2
--------------------------------------------------

❓ Question: What is the policy number for claim 1?
✅ Answer: 
Dear Mr. Anderson and Ms. Turner,

Thank you for reaching out to us regarding your claim. I understand your concern and the urgency you've expressed. I assure you that your claim is being handled with the utmost priority.

The policy number for your claim is PT567890.

To expedite the process, I have escalated your claim to our priority queue. Our claims team will provide you with a comprehensive update within the next 48 hours, as requested. If they require any additional information or documentation, they will co

In [16]:
# Let's read the exact summarization code from the workshop
with open('parasol-insurance/lab-materials/03/03-02-summarization.ipynb', 'r') as f:
    import json
    notebook = json.load(f)

# Print all code cells
for i, cell in enumerate(notebook['cells']):
    if cell['cell_type'] == 'code':
        print(f"\n--- Cell {i} ---")
        print(''.join(cell['source']))


--- Cell 2 ---
# Uncomment the following line only if you have not selected the right workbench image, or are using this notebook outside of the workshop environment.
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt

import json
import os
from os import listdir
from os.path import isfile, join

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

--- Cell 4 ---
# LLM Inference Server URL
inference_server_url = "http://granite-3-1-8b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLM definition
llm = ChatOpenAI(
    openai_api_key="EMPTY",   # Private model, we don't need a key
    openai_api_base=f"{inference_server_url}/v1",
    model_name="granite-3-1-8b-instruct",
    temperature=0.01,
    max_tokens=512,
   

In [17]:
# Feature 2 - SUMMARIZATION
# Using exact same approach as the workshop!

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# Use ChatOpenAI instead of VLLMOpenAI - same server!
chat_llm = ChatOpenAI(
    openai_api_key="EMPTY",
    openai_api_base=f"{inference_server_url}/v1",
    model_name="granite-3-1-8b-instruct",
    temperature=0.01,
    max_tokens=512,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
    top_p=0.9,
    presence_penalty=0.5,
    model_kwargs={"stream_options": {"include_usage": True}}
)

# Summary prompt template
summary_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        """You are a helpful insurance claims assistant at Parasol Insurance.
        Summarize the claim clearly in 5 bullet points covering:
        1. What happened
        2. Who was involved
        3. What damage occurred
        4. Claimant's emotional state
        5. What action is needed
        """),
    HumanMessagePromptTemplate.from_template(
        """### CLAIM:
        {input}
        ### SUMMARY:"""
    )
])

print("✅ Summarization setup ready!")
print("="*50)

# Summarize all 3 claims
for claim in claims:
    print(f"\n📄 Claim #{claim['claim-number']}: {claim['subject']}")
    print("-"*40)
    claim_input = f"Subject: {claim['subject']}\nContent:\n{claim['content']}"
    prompt = summary_template.invoke({"input": claim_input})
    resp = chat_llm.invoke(input=prompt)
    print("\n")
    print("="*50)

✅ Summarization setup ready!

📄 Claim #1: Claim for Recent Car Accident - Policy Number: AC-987654321
----------------------------------------
1. On January 2nd, 2024, at approximately 3:30 PM, Sarah Turner was driving her black Toyota Camry south on Birch Street in Evergreen when a blue Chevrolet Traverse, driven by Daniel Reynolds, ran a red light and collided with the front passenger side of her vehicle.
2. The individuals involved in the accident were Sarah Turner, the policyholder, and Daniel Reynolds, the driver of the other vehicle.
3. The damages sustained include extensive damage to the front bumper and right headlight of Sarah's Toyota Camry, as well as damages to the front driver's side of the Chevrolet Traverse.
4. Sarah Turner reported feeling shaken but otherwise unharmed following the accident. She promptly exchanged contact and insurance information with Daniel Reynolds and took photos of the accident scene and vehicle damages.
5. Sarah has submitted a police report, an

In [18]:
# Feature 3 - SENTIMENT ANALYSIS
sentiment_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        """You are an expert insurance claims analyst.
        Analyze the sentiment and urgency of this claim.
        Return exactly in this format:
        
        SENTIMENT: (Positive/Neutral/Negative)
        URGENCY: (Low/Medium/High/Critical)
        EMOTION: (Calm/Anxious/Frustrated/Angry)
        ACTION: (one sentence on what to do)
        """),
    HumanMessagePromptTemplate.from_template(
        """### CLAIM:
        {input}
        ### ANALYSIS:"""
    )
])

print("🎭 SENTIMENT ANALYSIS — All Claims")
print("="*50)

for claim in claims:
    print(f"\n📄 Claim #{claim['claim-number']}: {claim['subject']}")
    print("-"*40)
    claim_input = f"Subject: {claim['subject']}\nContent:\n{claim['content']}"
    prompt = sentiment_template.invoke({"input": claim_input})
    resp = chat_llm.invoke(input=prompt)
    print("\n")
    print("="*50)

🎭 SENTIMENT ANALYSIS — All Claims

📄 Claim #1: Claim for Recent Car Accident - Policy Number: AC-987654321
----------------------------------------
SENTIMENT: Neutral
URGENCY: Medium
EMOTION: Calm
ACTION: Review the attached documents and contact Sarah Turner for any additional information needed to process the claim.


📄 Claim #2: Urgent: Unacceptable Delay and Lack of Communication Regarding Claim #XYZ789
----------------------------------------
SENTIMENT: Negative
URGENCY: High
EMOTION: Angry
ACTION: Immediately address the claimant's concerns, provide a comprehensive update on the claim status, and expedite the processing of the claim to avoid further escalation.


📄 Claim #3: Urgent: Car Accident Claim Assistance Needed
----------------------------------------
SENTIMENT: Negative
URGENCY: High
EMOTION: Anxious
ACTION: Contact Jane Doe immediately to provide guidance on the next steps, including filing the insurance claim, obtaining repair estimates, and exchanging contact informat

In [19]:
# Final Summary - Show everything together!
print("🚗 InsureAI — Parasol Insurance Claims Assistant")
print("Built on Red Hat OpenShift AI")
print("By Madhu Ramakrishnappa")
print("="*50)

for claim in claims:
    print(f"\n{'='*50}")
    print(f"📄 CLAIM #{claim['claim-number']}")
    print(f"Subject: {claim['subject']}")
    print(f"{'='*50}")
    
    # RAG Answer
    print("\n🔍 RAG ANSWER:")
    result = rag_chain.invoke({
        "query": f"Summarize claim {claim['claim-number']} and who is at fault?"
    })
    print(result['result'])
    
    # Summary
    print("\n📝 SUMMARY:")
    claim_input = f"Subject: {claim['subject']}\nContent:\n{claim['content']}"
    prompt = summary_template.invoke({"input": claim_input})
    chat_llm.invoke(input=prompt)
    
    # Sentiment
    print("\n\n🎭 SENTIMENT:")
    prompt = sentiment_template.invoke({"input": claim_input})
    chat_llm.invoke(input=prompt)
    print("\n")

🚗 InsureAI — Parasol Insurance Claims Assistant
Built on Red Hat OpenShift AI
By Madhu Ramakrishnappa

📄 CLAIM #1
Subject: Claim for Recent Car Accident - Policy Number: AC-987654321

🔍 RAG ANSWER:

Dear Claimant,

Thank you for bringing this matter to our attention. I sincerely apologize for the frustration and inconvenience you have experienced with the processing of your claim. I understand that the lack of communication and updates has been disappointing, and I want to assure you that this is not the standard of service we strive to provide at Parasol Insurance.

Regarding the accident, based on the information provided in your claim, it appears that the other party involved in the collision is at fault. However, I must emphasize that this is a preliminary assessment, and our claims team is still in the process of thoroughly investigating the incident.

To address your concerns and expedite the claim process, I have taken the following actions:

1. I have escalated your claim to ou